## Recupero Storico degli Autori

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter
from pathlib import Path

In [2]:
current_dir = Path().resolve()
data_file = current_dir.parent / "Data" / "merged_authors.json"

with open(data_file, "r", encoding="utf-8") as f:
    merged_authors = json.load(f)

In [3]:
# Salvataggio di un dizionario con solo gli autori e l'anno+affiliazione
autori_istituzioni = {}

for autore, info in merged_authors.items():
    for collaborazioni, anno in info.items():
        autori_istituzioni[autore] = anno

In [4]:
with open('autori_istituzioni.json', 'w') as outfile:
    json.dump(autori_istituzioni, outfile)

Assumendo che se un'istituzione appare a tot anni di distanza il ricercatore sia rimasto affiliato a quell'istituzione per il tempo trascorso-> un po' semplicistica quindi ci starebbe di raffinare i parametri.

In [5]:
def calcola_anni_coperti(dizionario):
    anni_per_autore = {}
    
    for autore, storico in dizionario.items():
        anni = sorted([int(anno) for anno in storico.keys()])
        
        if len(anni) == 0:
            anni_per_autore[autore] = 0
            continue
        
        affiliazioni_range = {}
        
        for anno_str, affiliazioni in storico.items():
            anno = int(anno_str)
            for aff in affiliazioni:
                if aff not in affiliazioni_range:
                    affiliazioni_range[aff] = [anno, anno]
                else:
                    affiliazioni_range[aff][0] = min(affiliazioni_range[aff][0], anno)
                    affiliazioni_range[aff][1] = max(affiliazioni_range[aff][1], anno)
        
        anni_coperti = set()
        for aff, (primo_anno, ultimo_anno) in affiliazioni_range.items():
            for anno in range(primo_anno, ultimo_anno + 1):
                anni_coperti.add(anno)
        
        anni_per_autore[autore] = len(anni_coperti)
    
    return anni_per_autore


def stampa_statistiche(dizionario):
    anni_per_autore = calcola_anni_coperti(dizionario)
    
    print("Anni coperti per autore:")
    print("-" * 50)
    for autore, n_anni in anni_per_autore.items():
        print(f"{autore}: {n_anni} anni")
    
    print("\n" + "=" * 50)
    media = sum(anni_per_autore.values()) / len(anni_per_autore)
    print(f"In media per ogni autore sono coperti {media:.1f} anni")
    print("=" * 50)
    
    return anni_per_autore

In [6]:
risultati = stampa_statistiche(autori_istituzioni)

Anni coperti per autore:
--------------------------------------------------
Alessandro Liberati: 16 anni
Nicola De Stefano: 10 anni
Giulio Costantini: 8 anni
Elisa Maria Galliani: 9 anni
Niels Birbaumer: 18 anni
Giancarlo Comi: 3 anni
Massimo Filippi: 6 anni
María Trojano: 10 anni
Giacomo Rizzolatti: 16 anni
Emilio Perucca: 16 anni
Símone Rossi: 13 anni
Paolo Maria Rossini: 16 anni
Leonardo Pantoni: 2 anni
Stefano F. Cappa: 19 anni
Maria‐Luisa Gorno‐Tempini: 3 anni
J. Ogar: 2 anni
Bradley F. Boeve: 2 anni
Nina F. Dronkers: 2 anni
Bruce L. Miller: 2 anni
Giuseppe Jurman: 15 anni
Pietro Pietrini: 11 anni
Ele Ferrannini: 4 anni
Oliviero Bruni: 11 anni
L. Baldini: 15 anni
Guido Barbiellini: 22 anni
D. Bastieri: 15 anni
R. Bellazzini: 14 anni
G. F. Bignami: 13 anni
E. Bonamente: 1 anni
J. Bregeon: 1 anni
M. Brigida: 1 anni
P. A. Caraveo: 15 anni
E. Cavazzuti: 15 anni
Marco Ceccanti: 14 anni
C. Cecchi: 8 anni
S. Ciprini: 15 anni
S. Cutini: 15 anni
A. De Angelis: 9 anni
D. Fabiani: 4 anni
C. 

In [7]:
max = 0 
autor = ''
for autore, anni in risultati.items():
    if anni > max:
        max = anni
        autor = autore
    
print(f'Il numero massimo di anni coperti è di {max} da parte di {autor}')

Il numero massimo di anni coperti è di 26 da parte di Serenella Nardi


In [8]:
def scaglioni(dizionario):
    def categorizza(anni):
        if anni <= 1:
            return 'transient'
        elif anni < 5:
            return 'very_low'
        elif anni < 10:
            return 'low'
        elif anni < 20:
            return 'average'
        else:
            return 'high'
    
    categorie = Counter(categorizza(anni) for anni in dizionario.values())
    
    labels = {
        'transient': 'presente per un singolo anno.',
        'very_low': 'presente per meno di cinque anni.',
        'low': 'presente in un arco tra i cinque ed i dieci anni.',
        'average': 'presente per un arco tra i dieci ed i venti anni.',
        'high': 'presente per più di venti anni.'
    }
    
    totale = len(dizionario)
    
    print('Su un arco di venticinque anni analizzati:\n')
    for categoria in ['transient', 'very_low', 'low', 'average', 'high']:
        percentuale = (categorie[categoria] / totale) * 100
        print(f'Il {percentuale:.2f}% degli autori è {labels[categoria]}\n')

In [9]:
scaglioni(risultati)

Su un arco di venticinque anni analizzati:

Il 8.40% degli autori è presente per un singolo anno.

Il 47.23% degli autori è presente per meno di cinque anni.

Il 28.76% degli autori è presente in un arco tra i cinque ed i dieci anni.

Il 14.82% degli autori è presente per un arco tra i dieci ed i venti anni.

Il 0.78% degli autori è presente per più di venti anni.

